# NLP Document Similarity

**Course:** AIT 525

**Assignment:** NLP Document Similarity

**By:** Yashu Lanki

**Date:** 29 September, 2026

## Part 1: Preprocessing and Vectorization of Text Data

In [42]:
import pypdf
from pypdf import PdfReader

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [15]:
# Extract text from pdf files
def extract_text(pdf_path):
    reader = PdfReader(pdf_path)
    pages = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        pages.append(text)
    return "\n".join(pages)

In [25]:
# Apply function on the 2 pdf files
doc1_text = extract_text('nlp_data/Attention_Is_All_You_Need.pdf')
doc2_text = extract_text('nlp_data/BERT.pdf')

print(len(doc1_text),len(doc2_text))

39510 64140


In [19]:
# Tokenization, Stop words, Lemmatization, and Lowercasing
def preprocess(text):
    words = word_tokenize(text)
    tokens = []

    for word in words:
        if word.isalpha() and word.lower() not in stop_words:
            tokens.append(lemmatizer.lemmatize(word.lower()))
    return tokens

In [26]:
# Apply function to get tokens
doc1_tokens = preprocess(doc1_text)
doc2_tokens = preprocess(doc2_text)

print(doc1_tokens[:5])
print(doc2_tokens[:5])

['provided', 'proper', 'attribution', 'provided', 'google']
['bert', 'deep', 'bidirectional', 'transformer', 'language']


**Deliverables:**
- Python code that performs tokenization, stopword removal, and lemmatization/stemming on both documents.
- Provide a brief explanation of the preprocessing steps and how they help prepare text for vectorization.

**Answers:**

Tokenization breaks the text down into individual words. Stopword removal discards words that don't add much meaning (like "the" or "and"). Lemmatization reduces each word to its base/dictionary form (e.g., "running" → "run"), and lowercasing treats words like "Run" and "run" as the same word (Bird et al., 2009). These steps matter for vectorization because Bag of Words and TF-IDF turn each unique word into its own column — without lemmatization and lowercasing, "Run", "run", "running", and "runs" would each become separate columns instead of one, splitting what should be a single word's signal across several columns and inflating the vocabulary with noise. Stopword removal keeps that vocabulary focused on words that actually distinguish the documents' content, rather than filler words that appear everywhere.

## Part 2: Vector Representation of Documents

In [36]:
# Bag of Words
doc1_str = " ".join(doc1_tokens)
doc2_str = " ".join(doc2_tokens)

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform([doc1_str, doc2_str])
print(bow_matrix.shape)

(2, 2232)


In [37]:
# TF-IDF
tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer_matrix = tfidf_vectorizer.fit_transform([doc1_str, doc2_str])

print(tfidf_vectorizer_matrix.shape)

(2, 2232)


**Deliverables:**
- Python code that generates vector representations for both documents using BoW and TF-IDF.
- Provide a comparison of the two techniques (BoW vs. TF-IDF), explaining the strengths and weaknesses of each approach in capturing document similarities.

**Answers:**

Bag of Words counts each word's raw frequency in a document, while TF-IDF weighs those counts by how important a word is across the whole corpus (Jurafsky & Martin, 2025). This matters for capturing document similarity: a word that appears heavily in both papers just because it's common domain vocabulary (e.g., "model" or "layer") gets counted heavily by BoW even though it doesn't reflect real topical overlap, while TF-IDF downweights it precisely because it shows up in both documents. That said, with a corpus of only these 2 documents, TF-IDF's downweighting acts as a blunt on/off switch rather than a finely graded scale — a word's document frequency can only be 1 or 2, so it can't distinguish "somewhat common" from "extremely common" words the way it could over a large, hundred-document corpus (Pedregosa et al., 2011).

## Part 3: Document Similarity Calculation Using Cosine Similarity

In [43]:
bow_similarity = cosine_similarity(bow_matrix[0], bow_matrix[1])
tfidf_similarity = cosine_similarity(tfidf_vectorizer_matrix[0]
                                     ,tfidf_vectorizer_matrix[1])

In [47]:
print(f'BoW cosine similarity: {bow_similarity}')
print(f'Tfidf cosine similarity: {tfidf_similarity}')

BoW cosine similarity: [[0.51749606]]
Tfidf cosine similarity: [[0.44160527]]


**Deliverables:**
- Python code for calculating cosine similarity.
- An explanation of cosine similarity, including the formula and its significance.
- A comparison of cosine similarity scores obtained using BoW and TF-IDF, with insights into what the results indicate about document similarity.

**Answers:**

*Cosine similarity, formula and significance:* Cosine similarity measures the angle between two vectors, rather than their magnitude. The formula divides the dot product of the two vectors by the product of their magnitudes, which normalizes for length (Jurafsky & Martin, 2025). This matters for comparing the "Attention Is All You Need" and BERT papers specifically because they're different lengths — without normalizing, the longer BERT paper would likely produce a higher raw similarity score just because it has more words, regardless of how similar its actual content is. A score close to 1 means the two documents use words in a very similar pattern; a score close to 0 means very little overlap.

*BoW vs. TF-IDF similarity scores:* The TF-IDF cosine similarity (0.442) came out lower than the BoW score (0.517) because TF-IDF downweights words that appear in both documents — much of the shared vocabulary between these two papers (words like "model," "layer," "attention," and "training") is generic deep learning jargon rather than content genuinely unique to either paper, and BoW's raw counts don't distinguish between the two. Once that shared jargon is discounted, the remaining TF-IDF similarity reflects a moderate, not high, overlap — which fits the actual relationship between these papers: BERT builds directly on the Transformer architecture introduced in "Attention Is All You Need," but applies it to a different task (bidirectional language pretraining rather than machine translation), so the two papers share foundational vocabulary without being near-duplicates in content.

## Part 4: Evaluating and Interpreting Model Performance

**Deliverables:**
- A clear conclusion regarding the similarity of the two texts.
- Discuss how well the preprocessing and vectorization techniques work for capturing the meaning of the documents.

**Answers:**

Overall, the two papers are moderately similar. Both cosine similarity scores support this, and it matches their real relationship: BERT builds directly on the Transformer architecture from "Attention Is All You Need," but applies it to a different task. That said, both BoW and TF-IDF only capture surface-level word usage, not true meaning — they don't account for word order, synonyms, or context (Jurafsky & Martin, 2025), so two papers discussing closely related ideas in different terminology could appear less similar than they really are, and papers sharing jargon without deep conceptual overlap could appear more similar than they really are. Real-world noise added another limitation here: the PDF extraction for the Transformer paper picked up Google's boilerplate attribution notice before the actual title and abstract, which got counted as part of the document's vocabulary despite having nothing to do with its content. Overall, these techniques are useful for a quick, computationally cheap comparison, but they approximate meaning through word overlap rather than genuinely understanding it.

## References

Bird, S., Klein, E., & Loper, E. (2009). *Natural language processing with Python*. O'Reilly Media.

Jurafsky, D., & Martin, J. H. (2025). *Speech and language processing* (3rd ed. draft). https://web.stanford.edu/~jurafsky/slp3/

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, *12*, 2825–2830.